# Лабораторная работа №1
## Первичное исследование и оценка качества данных

**Тема:** анализ заказов небольшой сети кофеен.

В работе используется синтетический датасет `coffee_shop_sales.csv`. Он создан специально для учебного анализа и содержит числовые, категориальные и временные признаки, а также специально добавленные пропуски, дубликаты, неоднородные категории и выбросы.

# 1. Постановка задачи

### 1.1. Что это за данные
Датасет содержит информацию о заказах небольшой сети кофеен: дату заказа, товар, город, способ оплаты, канал продаж, размер заказа, цену, количество товаров, скидку, возраст клиента, рейтинг и время доставки.

### 1.2. Условный заказчик
Условным заказчиком анализа является управляющий сети кофеен. Ему важно понять структуру продаж, качество исходных данных и факторы, которые могут быть связаны с выручкой и оценками клиентов.

### 1.3. Возможные задачи интеллектуального анализа данных
1. Описательная аналитика: сравнение выручки по товарам, городам и каналам продаж.
2. Поиск аномалий: выявление необычно больших заказов, цен и времени доставки.
3. Сегментация заказов и клиентов по характеристикам покупок.

# 2. Паспорт датасета

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

data_path = Path("../data/coffee_shop_sales.csv")
if not data_path.exists():
    data_path = Path("data/coffee_shop_sales.csv")
df = pd.read_csv(data_path)

# Преобразуем дату в подходящий тип.
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

print(f"Количество строк: {df.shape[0]}")
print(f"Количество столбцов: {df.shape[1]}")

In [ ]:
print("Типы данных:")
display(df.dtypes.to_frame("dtype"))

print("\nПервые 5 строк:")
display(df.head())

### Краткая интерпретация признаков

- `Order_ID` — идентификатор заказа.
- `Order_Date` — дата заказа.
- `Product` — товар.
- `Category` — категория товара.
- `City` — город.
- `Payment` — способ оплаты.
- `Channel` — канал продаж.
- `Size` — размер заказа/позиции.
- `Unit_Price` — цена единицы товара.
- `Quantity` — количество единиц.
- `Discount` — размер скидки.
- `Customer_Age` — возраст клиента.
- `Rating` — оценка заказа.
- `Delivery_Min` — время доставки в минутах.
- `Revenue` — выручка по заказу.

# 3. Аудит качества данных
## 3.1. Пропуски

In [ ]:
missing = pd.DataFrame({
    "Количество": df.isna().sum(),
    "Доля_%": (df.isna().mean() * 100).round(2)
}).sort_values("Количество", ascending=False)

display(missing[missing["Количество"] > 0])

**Вывод.** В этом датасете больше всего пропусков в `Delivery_Min`: 662 (77.43%). Это частично объяснимо тем, что время доставки не относится к каждому каналу продаж. Пропуски в `Rating` и `City` нужно учитывать перед дальнейшим анализом.

## 3.2. Дубликаты

In [ ]:
full_duplicates = df.duplicated().sum()
id_duplicates = df["Order_ID"].duplicated().sum()

print("Полностью дублирующихся строк:", full_duplicates)
print("Повторяющихся Order_ID:", id_duplicates)

## 3.3. Типичные проблемы значений

In [ ]:
num_cols = ["Unit_Price", "Quantity", "Discount", "Customer_Age", "Rating", "Delivery_Min", "Revenue"]

display(df[num_cols].describe().T)

print("\nУникальные значения категориальных признаков:")
for col in ["Product", "City", "Payment", "Channel", "Size"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(15))

### Поиск подозрительных значений

В данных специально оставлены несколько больших значений `Quantity` и `Unit_Price`. Они не обязательно являются ошибками: это могут быть крупные заказы или специальные закупки, поэтому перед удалением их нужно проверять по исходному источнику.

## 3.4. Выбросы методом IQR

In [ ]:
x = df["Revenue"].dropna()

q1 = x.quantile(0.25)
q3 = x.quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df[(df["Revenue"] < lower) | (df["Revenue"] > upper)]

print(f"Q1 = {q1:.2f}")
print(f"Q3 = {q3:.2f}")
print(f"IQR = {iqr:.2f}")
print(f"Границы выбросов: [{lower:.2f}; {upper:.2f}]")
print(f"Количество выбросов: {len(outliers)}")

display(outliers[["Order_ID", "Product", "Quantity", "Unit_Price", "Revenue"]].head(10))

**Фактический результат:** для `Revenue` границы IQR составляют `-7.26` и `32.40`, а в зону выбросов попало **28 наблюдений**.

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(x=df["Revenue"])
plt.title("Boxplot выручки по заказам")
plt.xlabel("Revenue")
plt.tight_layout()
plt.show()

# 4. Разведочный анализ (EDA)

In [ ]:
# Распределение ключевого числового признака
plt.figure(figsize=(8, 4))
sns.histplot(df["Revenue"].dropna(), bins=30, kde=True)
plt.title("Распределение выручки по заказам")
plt.xlabel("Выручка")
plt.ylabel("Количество заказов")
plt.tight_layout()
plt.show()

**Комментарий к графику:** распределение выручки имеет правый хвост: большинство заказов сравнительно небольшие, но встречаются крупные заказы. Это согласуется с найденными методом IQR выбросами.

**Гипотеза:** крупные значения выручки могут быть связаны прежде всего с количеством товаров в заказе. Следующий шаг — проверить эту связь.

In [ ]:
# Частоты категориального признака
top_products = df["Product"].value_counts()

plt.figure(figsize=(8, 4))
sns.barplot(x=top_products.values, y=top_products.index)
plt.title("Количество заказов по товарам")
plt.xlabel("Количество заказов")
plt.ylabel("Товар")
plt.tight_layout()
plt.show()

**Комментарий к графику:** разные товары представлены в датасете неравномерно. Напитки встречаются чаще, чем некоторые категории еды.

**Гипотеза:** структура ассортимента может влиять на общую выручку, поэтому полезно сравнить не только количество заказов, но и среднюю выручку по товарным категориям.

In [ ]:
# Связь двух признаков
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="Quantity",
    y="Revenue",
    hue="Category",
    alpha=0.65
)
plt.title("Связь количества товаров и выручки")
plt.xlabel("Количество")
plt.ylabel("Выручка")
plt.tight_layout()
plt.show()

**Комментарий к графику:** между количеством товаров и выручкой наблюдается ожидаемая положительная связь: при большем количестве позиций выручка обычно выше. При этом точки сильно различаются из-за разных цен товаров и скидок.

**Вопрос:** насколько сильно цена единицы товара влияет на выручку при одинаковом количестве? Это можно проверить на следующем этапе анализа.

# 5. Типы признаков и шкалы измерения

| Признак | Шкала | Обоснование |
|---|---|---|
| `Order_ID` | Номинальная | Идентификатор: числовые операции над ним не имеют содержательного смысла. |
| `Order_Date` | Интервальная | Разности между датами и длительности имеют смысл; календарный ноль условен. |
| `Product` | Номинальная | Названия товаров не имеют естественного порядка. |
| `Category` | Номинальная | Категории товаров не упорядочены. |
| `City` | Номинальная | Названия городов — категории. |
| `Payment` | Номинальная | Способы оплаты не упорядочены. |
| `Channel` | Номинальная | Каналы продаж — категории. |
| `Size` | Порядковая | Размеры Small < Medium < Large имеют естественный порядок, но расстояния между ними не измерены. |
| `Unit_Price` | Шкала отношений | Есть содержательный ноль; отношения цен имеют смысл. |
| `Quantity` | Шкала отношений | Количество имеет естественный ноль, можно сравнивать во сколько раз оно больше. |
| `Discount` | Шкала отношений | Нулевая скидка означает отсутствие скидки. |
| `Customer_Age` | Шкала отношений | Возраст имеет естественный ноль; отношения возрастов содержательны. |
| `Rating` | Порядковая | Оценки упорядочены, но равенство расстояний между соседними оценками нельзя гарантировать без описания шкалы. |
| `Delivery_Min` | Шкала отношений | Ноль минут имеет смысл; длительности можно сравнивать отношениями. |
| `Revenue` | Шкала отношений | Нулевая выручка означает отсутствие дохода; допустимы отношения значений. |

### Сравнение двух признаков и агрегаты

**`Product` (номинальная шкала).** Среднее и медиана не имеют смысла: названия товаров нельзя складывать или упорядочивать. Мода корректна — она показывает наиболее часто встречающийся товар. Частоты и доли категорий также корректны.

**`Size` (порядковая шкала).** Медиана и мода допустимы, если категории упорядочены как Small < Medium < Large. Среднее арифметическое кодов 1, 2, 3 методологически сомнительно: оно предполагает равные расстояния между размерами, которых данные не задают. Лучше показывать частоты, моду и медиану категории.

В pandas технически можно закодировать категории числами и вычислить среднее, но возможность вычисления не означает содержательную корректность результата.

# 6. Пропуски: MCAR / MAR / MNAR

Для признака `Rating` рассматриваются три альтернативные гипотезы.

1. **MCAR:** пропуск не зависит ни от одного наблюдаемого или ненаблюдаемого признака. Проверка: сравнить долю пропусков по `City`, `Channel`, `Product` и другим доступным признакам. Если заметных различий нет, гипотеза MCAR становится более правдоподобной.
2. **MAR:** пропуск связан с наблюдаемыми признаками. Например, рейтинг может чаще отсутствовать в определённом канале продаж. Проверка ниже сравнивает долю пропусков по `Channel` и `City`.
3. **MNAR:** вероятность пропуска зависит от самого значения рейтинга или от ненаблюдаемой причины, например от нежелания клиента оставлять плохой отзыв. По имеющимся столбцам такую гипотезу нельзя надёжно подтвердить; нужна дополнительная информация о процессе сбора отзывов.

Таким образом, по этому датасету можно непосредственно проверить признаки MAR и поискать косвенные различия, но строго различить MCAR и MNAR только по наблюдаемым данным нельзя.

In [ ]:
# Индикатор пропуска рейтинга
# 1 означает, что рейтинг отсутствует.
df["Rating_missing"] = df["Rating"].isna().astype(int)

print("Доля пропусков рейтинга по каналу продаж:")
display(
    df.groupby("Channel", dropna=False)["Rating_missing"]
      .mean()
      .mul(100)
      .round(2)
      .to_frame("Missing_Rating_%")
)

print("Доля пропусков рейтинга по городу:")
display(
    df.groupby("City", dropna=False)["Rating_missing"]
      .mean()
      .mul(100)
      .round(2)
      .to_frame("Missing_Rating_%")
)

### Задание 3. Индикатор пропуска как источник информации

Индикатор `Rating_missing` равен 1, если рейтинг не заполнен, и 0 в противном случае. Сам факт пропуска несёт информацию, когда отсутствие значения связано с процессом наблюдения или с характеристиками заказа. Например, клиенты одного канала продаж могут реже оставлять оценки.

**Почему значимость индикатора важна.** Если после импутации индикатор пропуска остаётся важным предиктором целевой переменной, это может означать, что модель использует информацию о механизме пропуска. Это не доказывает ошибочность импутации само по себе, но служит сигналом проверить её: простая подстановка среднего/медианы могла стереть различия между группами или создать искусственную структуру. Нужно сравнить модели с индикатором и без него, проверить утечку целевой информации и обосновать механизм пропусков.

**Примеры:**
- **Полезен:** `Rating_missing` — отсутствие оценки может быть связано с каналом заказа или опытом клиента и помогать описать процесс сбора обратной связи.
- **Бесполезен:** индикатор пропуска столбца, в котором вообще нет пропусков. Он всегда равен 0 и не несёт вариации.
- **Опасен:** индикатор пропуска признака, который отсутствует из-за процесса, связанного с целевой переменной или будущим событием. Использование такого индикатора может привести к утечке данных и завышенной оценке качества модели.

**Ограничение:** по одним наблюдаемым данным нельзя окончательно доказать MCAR или MNAR. Нужны сведения о том, как собирались оценки и почему часть клиентов их не оставила.

# 7. Выброс — ошибка или сигнал?

Для `Revenue` граница верхних выбросов определяется методом IQR. В качестве примера ниже автоматически выбирается наблюдение с наибольшей выручкой среди найденных выбросов.

Возможные реальные причины:
- действительно крупный заказ или покупка нескольких единиц товара;
- оптовая или корпоративная покупка;
- ошибка ввода количества или цены;
- некорректно применённая скидка или ошибка расчёта выручки.

**Сравнение вариантов обработки:**
1. **Удалить выброс.** Уместно только после подтверждения, что это ошибка или наблюдение вне области исследования. Иначе теряется информация о крупных заказах и средняя выручка может быть занижена.
2. **Оставить без изменений.** Правильный выбор, если значение реально и важно для бизнеса. Следует использовать устойчивые статистики и отдельно анализировать крупные заказы.
3. **Преобразовать.** Логарифмирование или другое преобразование может уменьшить влияние правого хвоста при моделировании, но меняет интерпретацию шкалы. Оно не исправляет ошибочные исходные данные.

**Вывод:** нельзя автоматически считать статистический выброс ошибкой. Сначала нужно проверить исходный заказ и правила расчёта `Revenue`; до такой проверки наблюдение лучше сохранить и явно отметить.

In [ ]:
one_outlier = outliers.sort_values("Revenue", ascending=False).head(1)
display(one_outlier[["Order_ID", "Product", "Quantity", "Unit_Price", "Discount", "Revenue"]])

# 8. Преобразование или винзоризация?

Для `Revenue` наблюдается правосторонняя асимметрия. Возможные подходы:

| Подход | Когда предпочтителен | Влияние на интерпретацию и модели |
|---|---|---|
| Логарифмирование `log1p(x)` | Сильная правая асимметрия, значения неотрицательные | Сжимает большие значения сильнее малых; может уменьшить влияние крупных заказов в линейной модели. Коэффициенты относятся к логарифму выручки, поэтому интерпретация меняется. |
| Квадратный корень `sqrt(x)` | Умеренная асимметрия и неотрицательные значения | Сжимает хвост мягче логарифма; интерпретация становится менее прямой, но преобразование обычно проще объяснить. |
| Винзоризация | Экстремальные значения чрезмерно влияют на анализ, но строки нельзя удалять | Значения за выбранными квантилями заменяются пороговыми. Сохраняется число наблюдений, но искажается величина экстремумов и теряется часть информации. |

Для графиков исходное распределение полезно оставить как есть, а преобразованное показать отдельно: так видно, насколько уменьшилась асимметрия. Для линейных моделей преобразование может улучшить поведение остатков, но это нужно проверять на диагностике модели, а не считать гарантированным.

В этом датасете `Revenue` положительна, поэтому можно сравнить исходную выручку, `log1p(Revenue)`, `sqrt(Revenue)` и винзоризацию по 1-му и 99-му процентилям. Винзоризованные значения следует хранить в отдельном столбце, не перезаписывая исходные данные.

In [ ]:
# Сравнение преобразований для неотрицательной выручки
revenue = df["Revenue"].dropna()
comparison = pd.DataFrame({
    "Исходная выручка": revenue,
    "Логарифм log1p": np.log1p(revenue),
    "Квадратный корень": np.sqrt(revenue),
    "Винзоризация 1%-99%": revenue.clip(
        lower=revenue.quantile(0.01),
        upper=revenue.quantile(0.99)
    )
})
display(comparison.agg(["mean", "median", "skew"]).round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(revenue, bins=25, ax=axes[0])
axes[0].set_title("Исходная выручка")
sns.histplot(np.log1p(revenue), bins=25, ax=axes[1])
axes[1].set_title("log1p(выручка)")
sns.histplot(np.sqrt(revenue), bins=25, ax=axes[2])
axes[2].set_title("sqrt(выручка)")
plt.tight_layout()
plt.show()

# 9. Средние, которые могут вводить в заблуждение

Для `Revenue` полезно сравнить среднее и медиану. Если среднее заметно больше медианы, это будет признаком правосторонней асимметрии: несколько крупных заказов тянут среднее вверх.

Геометрическое среднее для положительных значений `Revenue` вычислить можно, но интерпретировать его нужно осторожно. Для описания «типичного» заказа медиана часто оказывается устойчивее к крупным выбросам.

In [ ]:
revenue = df["Revenue"].dropna()

print("Среднее:", round(revenue.mean(), 2))
print("Медиана:", round(revenue.median(), 2))
print("Геометрическое среднее:", round(np.exp(np.log(revenue).mean()), 2))

**Вывод по средним значениям.** Среднее чувствительно к крупным заказам и обычно выше медианы при правосторонней асимметрии. Медиана описывает центральный заказ и устойчивее к отдельным экстремальным значениям. Геометрическое среднее применимо к положительным значениям, но отражает скорее мультипликативный центр и обычно ниже арифметического среднего.

## 10.1. Неправильная диаграмма

Для частот товаров в работе построена столбчатая диаграмма. Для этих категориальных данных линейный график был бы хуже: линия соединяет соседние категории и визуально создаёт впечатление непрерывности или естественного порядка между названиями товаров, хотя такого порядка нет.

Такой график может заставить зрителя ошибочно решить, что между соседними товарами существует последовательный рост или спад, а положение товара на оси X имеет количественный смысл. Для сравнения отдельных категорий корректнее использовать столбчатую диаграмму, где длина столбца напрямую показывает частоту.

## 10.2. Одна информация — разные графики

Для пары `Category` и `Revenue` можно использовать два графика:

1. **Boxplot** — показывает медиану, квартильный разброс и выбросы внутри каждой категории.
2. **Barplot среднего** — позволяет быстро сравнить среднюю выручку категорий.

Boxplot лучше подходит для изучения распределения, а barplot — для компактного сравнения центральных значений.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="Category", y="Revenue")
plt.title("Распределение выручки по категориям")
plt.xlabel("Категория")
plt.ylabel("Выручка")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=df, x="Category", y="Revenue", estimator="mean", errorbar="sd")
plt.title("Средняя выручка по категориям")
plt.xlabel("Категория")
plt.ylabel("Средняя выручка")
plt.tight_layout()
plt.show()

**Сравнение визуализаций.** Boxplot показывает медиану, межквартильный размах и потенциальные выбросы внутри каждой категории. Столбчатая диаграмма средних значений компактно сравнивает среднюю выручку, но скрывает разброс и может быть чувствительна к экстремальным заказам. Поэтому для первичного анализа распределений предпочтительнее boxplot, а barplot использовать как дополнительное сравнение средних.